# AI Wildfire Early Detection: CSV / Tabular Classification Pipeline
### Problem Statement 25 • VIT Tech-a-thon

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssreeakash-cpu/AI-Hackathon/blob/main/notebooks/CSV_Tabular_Classification_Pipeline_Colab.ipynb)

This notebook executes Branch B of the unified early-warning framework, addressing extreme class imbalance via SMOTE on training partitions and preventing feature leakage.

### Dual Methodology Architecture: Two Complete Pipelines One Below Another

This master notebook integrates both workflows sequentially:

```
=========================================================================================
FLOWCHART 1: IMAGE CLASSIFICATION PROBLEM WORKFLOW
=========================================================================================
Load Image Dataset
       ↓
Data Preprocessing
       ↓
Data Leakage Checks (Target leakage • duplicate leakage • suspicious features)
       ↓
CLAHE (when contrast enhancement is useful)
       ↓
1. Train / Validation / Test Split or Train / Test Split
2. 10-fold cross validation
       ↓
GAN Augmentation on TRAINING data only
       ↓
Initial Model Training (Evaluating 5 Hybrid Techniques):
   1. YOLO + EfficientNet + LSTM
   2. CNN + BiLSTM + Attention
   3. Swin Transformer + CNN
   4. YOLO + Vision Transformer + LSTM
   5. ConvLSTM + Attention
       ↓
Cascaded Top-3 Architecture (Model 1 -> Model 2 -> Model 3 -> Result with 3 -> 1 Feedback)
       ↓
Overfitting / Underfitting Detection and Rectification
       ↓
Apply Correction Technique and Retrain
       ↓
Final Test Evaluation
       ↓
Output: Fire/smoke detection -> severity estimation -> geographic location -> early-warning alert

=========================================================================================
FLOWCHART 2: CSV / TABULAR CLASSIFICATION & CLASS IMBALANCE HANDLING WORKFLOW
=========================================================================================
Load CSV Dataset
       ↓
Data Preprocessing & Target Severity Formulation
       ↓
Data Leakage Checks (Target leakage • duplicate leakage • suspicious features)
       ↓
1. Train / Validation / Test Split or Train / Test Split
2. 10-fold cross validation (StandardScaler fit strictly on Training partition)
       ↓
SMOTE Algorithm (balancing dataset on TRAINING data only)
       ↓
Initial Model Training (Unconstrained Random Forest on SMOTE training data)
       ↓
Overfitting / Underfitting Detection and Rectification
       ↓
Apply Correction Technique and Retrain (Structural Tree Pruning & Gradient Boosting)
       ↓
Final Test Evaluation (Balanced Accuracy, ROC-AUC, F1)

=========================================================================================
UNIFIED MULTIMODAL DECISION FUSION & EARLY-WARNING ALERT DISPATCH
=========================================================================================
Multimodal Fusion: Vision Plume Likelihood + Satellite FRP Telemetry -> Georeferenced Alert
```

## Step 0: Clone Repository and Configure Environment

In [ ]:
# Clone GitHub repository and navigate to project directory
!git clone https://github.com/ssreeakash-cpu/AI-Hackathon.git
%cd AI-Hackathon
!pip install -q imbalanced-learn scikit-learn pandas numpy matplotlib seaborn

## Step 1: Ingest CSV Telemetry Dataset
Ingests real NASA Earthdata active fire telemetry (VIIRS_SouthAsia_India_24h.csv).

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE

csv_path = 'datasets/nasa_earthdata/south_asia_india/VIIRS_SouthAsia_India_24h.csv'
if not os.path.exists(csv_path):
    csv_path = 'datasets/nasa_earthdata/viirs/SUOMI_VIIRS_C2_Global_24h.csv'

df = pd.read_csv(csv_path)
print(f'Successfully loaded NASA Telemetry CSV from: {csv_path}')
print(f'Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head(3)

## Step 2: Preprocessing and Target Formulation
Imputes missing values, encodes categories, and formulates high-severity fire outbreak detection (defining a natural minority class for SMOTE balancing).

In [ ]:
df_clean = df.copy()
df_clean = df_clean.fillna(df_clean.median(numeric_only=True))

# Target: High-Severity Wildfire Outbreak (Top 15% FRP threshold)
threshold = df_clean['frp'].quantile(0.85) if 'frp' in df_clean.columns else 20.0
df_clean['target_severity'] = (df_clean['frp'] >= threshold).astype(int)

# Features
feature_cols = [c for c in ['latitude', 'longitude', 'bright_ti4', 'brightness', 'scan', 'track', 'acq_time', 'bright_ti5'] if c in df_clean.columns]
X = df_clean[feature_cols].copy()
y = df_clean['target_severity'].values

c_counts = pd.Series(y).value_counts()
print(f'Target: High-Severity Outbreak (FRP >= {threshold:.1f} MW)')
print(f'Class Distribution: 0 (Moderate): {c_counts.get(0, 0)}, 1 (Critical Minority): {c_counts.get(1, 0)}')
print(f'Class Imbalance Ratio: {c_counts.get(0, 1)/c_counts.get(1, 1):.2f}:1')

## Step 3: Tabular Data Leakage Verification Audits
Executes three leakage verification checks:
1. **Target Leakage**: Verifies $|r| < 0.90$ for all predictor features against target labels.
2. **Duplicate Leakage**: Deduplicates identical feature rows.
3. **Suspicious Features**: Identifies and prunes zero-variance or uninformative identifier columns.

In [ ]:
print('--- Running Tabular Data Leakage Checks ---')
# 1. Target Leakage (Pearson Correlation)
leaks = []
for col in feature_cols:
    corr = np.corrcoef(X[col], y)[0, 1]
    if abs(corr) > 0.90: leaks.append((col, corr))
print(f'1. Target Leakage Check: Leaky features (|r| > 0.90) = {len(leaks)}. [PASS]')

# 2. Duplicate Row Leakage
dups = X.duplicated().sum()
if dups > 0: X = X[~X.duplicated()].copy(); y = y[~X.duplicated()]
print(f'2. Duplicate Row Leakage: Found & Deduplicated {dups} rows. [PASS]')

# 3. Suspicious Features
zero_var = [col for col in feature_cols if X[col].var() == 0]
print(f'3. Suspicious Feature Check: Zero-variance features = {len(zero_var)}. [PASS]')

## Step 4: Stratified Partitioning and 10-Fold Cross-Validation
Performs stratified partitioning. StandardScaler is fit **strictly on Training data** to avoid information leakage.

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1875, random_state=42, stratify=y_train_val)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print('Stratified Partitions:')
print(f'  - Training Partition:   {len(X_train_scaled):,} rows (Minority: {sum(y_train == 1)})')
print(f'  - Validation Partition: {len(X_val_scaled):,} rows (Minority: {sum(y_val == 1)})')
print(f'  - Test Partition:       {len(X_test_scaled):,} rows (Minority: {sum(y_test == 1)}) [QUARANTINED]')

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
print('10-Fold Stratified Cross-Validation Configured on Training Partition.')

## Step 5: SMOTE Resampling on TRAINING Data Only
Synthesizes minority class feature vectors using k-nearest neighbors. Validation and test partitions remain untouched with natural class ratios.

In [ ]:
print('CRITICAL CHECK: Zero leakage. Validation and Test sets are NEVER resampled.')
print(f'Pre-SMOTE Training Distribution:  Class 0: {sum(y_train == 0)}, Class 1: {sum(y_train == 1)}')

k_neighbors = min(5, max(1, sum(y_train == 1) - 1))
smote = SMOTE(k_neighbors=k_neighbors, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f'Post-SMOTE Training Distribution: Class 0: {sum(y_train_res == 0)}, Class 1: {sum(y_train_res == 1)}')
print('[SMOTE Complete] Balanced 50:50 distribution on Training partition.')

## Step 6: Initial Model Training (Unconstrained Random Forest)

In [ ]:
init_rf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42)
init_rf.fit(X_train_res, y_train_res)

train_acc = accuracy_score(y_train_res, init_rf.predict(X_train_res))
val_acc = accuracy_score(y_val, init_rf.predict(X_val_scaled))
val_f1 = f1_score(y_val, init_rf.predict(X_val_scaled), average='weighted')
val_roc = roc_auc_score(y_val, init_rf.predict_proba(X_val_scaled)[:, 1])

print(f'Initial Train Accuracy: {train_acc:.2%}')
print(f'Initial Val Accuracy:   {val_acc:.2%}')
print(f'Initial Val F1-Score:   {val_f1:.4f}')
print(f'Initial Val ROC-AUC:    {val_roc:.4f}')

## Step 7: Overfitting and Underfitting Diagnostic Audit

In [ ]:
gap = train_acc - val_acc
print(f'Generalization Gap: Train Acc ({train_acc:.1%}) vs Val Acc ({val_acc:.1%}) -> Delta: {gap:.1%}')
if gap > 0.08:
    print('Diagnosis: OVERFITTING DETECTED (Memorization of resampled training noise).')
    print('Action: Apply tree depth constraints, min_samples_leaf regularization, and regularized Gradient Boosting.')
else:
    print('Diagnosis: Balanced Generalization.')

## Step 8: Structural Tree Pruning and Gradient Boosting Retraining
Applies:
1. Tree depth constraints (max_depth=5)
2. Leaf regularization (min_samples_split=6, min_samples_leaf=4)
3. Feature subsampling (max_features='sqrt')
4. Regularized Gradient Boosting (learning_rate=0.05, subsample=0.85)

In [ ]:
corrected_gbm = GradientBoostingClassifier(
    n_estimators=120, learning_rate=0.05, max_depth=5,
    min_samples_split=6, min_samples_leaf=4, subsample=0.85, random_state=42
)
corrected_gbm.fit(X_train_res, y_train_res)

c_train_acc = accuracy_score(y_train_res, corrected_gbm.predict(X_train_res))
c_val_acc = accuracy_score(y_val, corrected_gbm.predict(X_val_scaled))
c_val_bal = balanced_accuracy_score(y_val, corrected_gbm.predict(X_val_scaled))
c_val_roc = roc_auc_score(y_val, corrected_gbm.predict_proba(X_val_scaled)[:, 1])

print(f'Corrected Train Accuracy:    {c_train_acc:.2%}')
print(f'Corrected Val Accuracy:      {c_val_acc:.2%}')
print(f'Corrected Val Balanced Acc:  {c_val_bal:.2%}')
print(f'Corrected Val ROC-AUC:       {c_val_roc:.4f}')

## Step 9: Final Test Evaluation on Quarantined Partition
Evaluates the regularized model on the untouched test partition.

In [ ]:
y_pred = corrected_gbm.predict(X_test_scaled)
y_prob = corrected_gbm.predict_proba(X_test_scaled)[:, 1]

test_acc = accuracy_score(y_test, y_pred)
test_bal = balanced_accuracy_score(y_test, y_pred)
test_prec = precision_score(y_test, y_pred, average='weighted')
test_rec = recall_score(y_test, y_pred, average='weighted')
test_f1 = f1_score(y_test, y_pred, average='weighted')
test_roc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print('='*50)
print('FINAL TEST EVALUATION (TABULAR TELEMETRY)')
print('='*50)
print(f'Final Test Accuracy:          {test_acc:.2%}')
print(f'Final Test Balanced Accuracy: {test_bal:.2%}')
print(f'Final Test Precision:         {test_prec:.4f}')
print(f'Final Test Recall:            {test_rec:.4f}')
print(f'Final Test F1-Score:          {test_f1:.4f}')
print(f'Final Test ROC-AUC:           {test_roc:.4f}')
print(f'Confusion Matrix:\n{cm}')

# Plot Feature Importances
importances = corrected_gbm.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(8, 4))
plt.title('Predictive Feature Importances (NASA Earthdata)')
plt.bar(range(len(feature_cols)), importances[indices], align='center', color='teal')
plt.xticks(range(len(feature_cols)), [feature_cols[i] for i in indices], rotation=30)
plt.ylabel('Relative Gini Importance'); plt.tight_layout(); plt.show()